# Sound generation — sixteen impossible voices

Use a **T4 GPU**. This notebook uses the pre-trained [AudioLDM small full v2](https://huggingface.co/cvssp/audioldm-s-full-v2) diffusion model. It requires no Stable Audio gated-model login. Reference: [Diffusers AudioLDM pipeline](https://huggingface.co/docs/diffusers/api/pipelines/audioldm). It generates sixteen unique prompts, with durations from 2 to 9.5 seconds. The language text is separate; these calls are not speech or phrase readings.

Listen to every result. Check that calls differ, have no music or intelligible speech, and suit the imagined creature. Regenerate weak calls with alternate seeds; increase inference steps and compare. Save this executed notebook using File → Download → Download .ipynb. No training of the audio model is claimed.


This is the clean replay notebook for the completed audio pass. Actual prompts, seeds, generation times, and durations are in sound-run.json. Names may be curated after generation; specimen IDs preserve correspondence.

Each non-silent float waveform is peak-normalized to 0.85 before 16-bit encoding, so quiet calls remain audible without clipping.

This is the clean replay notebook for the completed audio pass. Actual prompts, seeds, generation times, and durations are in sound-run.json. Names may be curated after generation; specimen IDs preserve correspondence.

In [ ]:
# Pre-trained diffusion audio: AudioLDM, separate from image training.
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','diffusers==0.35.1','transformers==4.56.2','accelerate==1.10.1','soundfile','sentencepiece'])
import json, gc, time
from pathlib import Path
import numpy as np
import torch
import soundfile as sf
from diffusers import AudioLDMPipeline
from IPython.display import Audio, display
assert torch.cuda.is_available(), 'Select a T4 GPU first.'
ROOT=Path('/content/postcards-from-my-jungle');ROOT.mkdir(exist_ok=True)
(ROOT/'sounds').mkdir(exist_ok=True)
ANIMALS=json.loads(r'''[
  {
    "id": 1,
    "name": "Fernfox",
    "latin": "Vulpes frondis",
    "habitat": "The moss understory",
    "description": "A quiet keeper of damp places. It gathers fallen light in its feathery ruff, then leaves a little behind for anything afraid of the dark.",
    "phrase": "Mi luma na.",
    "translation": "I carry light home.",
    "soundPrompt": "soft fox chirps mixed with fluttering moth wings, three gentle rising calls",
    "duration": 2
  },
  {
    "id": 2,
    "name": "Glasswhisker",
    "latin": "Felis vitrea",
    "habitat": "The upper canopy",
    "description": "A silver-faced wanderer whose translucent whiskers catch the colour of whatever it passes. When the rain stops, its soft coat seems to remember the sound.",
    "phrase": "Ti shiri ru.",
    "translation": "You sing with the rain.",
    "soundPrompt": "delicate glassy insect trills and short airy wing flutters",
    "duration": 2.5
  },
  {
    "id": 3,
    "name": "Velvet Croaker",
    "latin": "Felis velutina",
    "habitat": "The stillwater pools",
    "description": "Round as a river stone, soft as a fern. It answers distant thunder with a voice much larger than its small body.",
    "phrase": "Mi moru ru.",
    "translation": "I listen to the rain.",
    "soundPrompt": "deep hollow frog croaks, slow bubbling pulses with wet resonant texture",
    "duration": 3
  },
  {
    "id": 4,
    "name": "Lantern Lynx",
    "latin": "Lynx lucerna",
    "habitat": "The lantern grove",
    "description": "Its belly glows like the last warm window in a sleeping village. Other creatures follow it home without ever asking the way.",
    "phrase": "Luma na sa.",
    "translation": "Light and home are together.",
    "soundPrompt": "soft fluttering, tiny warm bell-like peeps, irregular gentle pulses",
    "duration": 3.5
  },
  {
    "id": 5,
    "name": "Rootback",
    "latin": "Felis radix",
    "habitat": "The root tangle",
    "description": "A patient creature whose rough coat shelters a miniature forest. Nobody remembers where the animal ends and the garden begins.",
    "phrase": "Mi na moru.",
    "translation": "I listen at home.",
    "soundPrompt": "low woody creaking calls, soft turtle grunts and dry bark clicks",
    "duration": 4
  },
  {
    "id": 6,
    "name": "Cloud Lemur",
    "latin": "Lemur nebula",
    "habitat": "The mist canopy",
    "description": "It rests in the highest branches and knots the morning fog into little nests. Its tail is always the last thing to disappear.",
    "phrase": "Mi shiri luma.",
    "translation": "I sing to the light.",
    "soundPrompt": "airy lemur coos, breathy rising whistles and a faint purr",
    "duration": 4.5
  },
  {
    "id": 7,
    "name": "Reedrunner",
    "latin": "Felis arundinis",
    "habitat": "The reed margins",
    "description": "All legs, almost no weight. It moves across the marsh so lightly that the water notices only after it has gone.",
    "phrase": "Ti ru na.",
    "translation": "You bring rain home.",
    "soundPrompt": "quick reed bird chirps, hollow beak clicks, staccato watery tweets",
    "duration": 5
  },
  {
    "id": 8,
    "name": "Puddle Lynx",
    "latin": "Lynx pluvia",
    "habitat": "The rain gardens",
    "description": "It watches its reflection until the reflection blinks first. Its whiskers ripple whenever a storm is still a long way off.",
    "phrase": "Mi moru ti.",
    "translation": "I am listening to you.",
    "soundPrompt": "raspy kitten chatter and soft liquid purring with isolated droplets",
    "duration": 5.5
  },
  {
    "id": 9,
    "name": "Moonmuzzle",
    "latin": "Felis lunaris",
    "habitat": "The moonlit clearing",
    "description": "A crescent of pale fur curls around its muzzle. Every evening it tells the moon what happened while she was away.",
    "phrase": "Luma shiri sa.",
    "translation": "The lights sing together.",
    "soundPrompt": "low rounded bird hoots followed by two mellow flute whistles",
    "duration": 6
  },
  {
    "id": 10,
    "name": "Sporemouse",
    "latin": "Mus spora",
    "habitat": "The mushroom floor",
    "description": "Smaller than a curled leaf, it sleeps inside abandoned seed pods. A trail of tiny glowing footprints gives its hiding place away.",
    "phrase": "Mi na luma.",
    "translation": "My home is bright.",
    "soundPrompt": "tiny high mouse squeaks and soft mushroom-like popping clicks",
    "duration": 6.5
  },
  {
    "id": 11,
    "name": "Coralcrown",
    "latin": "Felis corallium",
    "habitat": "The flooded orchard",
    "description": "Its crown of moss branches blooms after every rainfall. Beneath the orchard trees, it waits for the water to remember its name.",
    "phrase": "Ru luma sa.",
    "translation": "Rain and light meet.",
    "soundPrompt": "gentle deer bleats with hollow underwater gurgles and soft snorts",
    "duration": 7
  },
  {
    "id": 12,
    "name": "Echo Sloth",
    "latin": "Bradypus sonorus",
    "habitat": "The hanging gardens",
    "description": "It takes a whole afternoon to answer a greeting. The forest is patient enough to wait, and always glad when it finally replies.",
    "phrase": "Mi moru sa.",
    "translation": "I listen with everyone.",
    "soundPrompt": "very slow breathy sloth calls, soft descending hums and distant echoes",
    "duration": 7.5
  },
  {
    "id": 13,
    "name": "Thornwhisker",
    "latin": "Felis spina",
    "habitat": "The blackwater stream",
    "description": "It waits at the blackwater edge with its whiskers folded like letters. At dusk it touches the ripples and listens for a reply.",
    "phrase": "Ru shiri na.",
    "translation": "Rain sings of home.",
    "soundPrompt": "submerged croaks, gentle water bubbles, low fish-like drumming pulses",
    "duration": 8
  },
  {
    "id": 14,
    "name": "Bramble Bat",
    "latin": "Vesper rubus",
    "habitat": "The tangled hollow",
    "description": "Wrapped in its own wings, it resembles a small sleeping fruit. It navigates by asking the darkness very polite questions.",
    "phrase": "Ti luma mi.",
    "translation": "You are my light.",
    "soundPrompt": "short bat-like chirps shifted into audible range, dry fluttering wings",
    "duration": 8.5
  },
  {
    "id": 15,
    "name": "Dewkeeper",
    "latin": "Felis ros",
    "habitat": "The fern spiral",
    "description": "A quiet collector of morning. The dew on its pale coat stays cool until sunset, when it shares a sip with the roots.",
    "phrase": "Mi ru sa.",
    "translation": "I share the rain.",
    "soundPrompt": "delicate wet clicks, soft sliding squeaks and rounded tiny bubbles",
    "duration": 9
  },
  {
    "id": 16,
    "name": "Hushbear",
    "latin": "Ursus quietus",
    "habitat": "The far green",
    "description": "The largest creature in the archive has the smallest voice. It sings only when the rest of the jungle has fallen asleep.",
    "phrase": "Na moru mi.",
    "translation": "Home is listening for me.",
    "soundPrompt": "deep gentle bear murmurs, warm breathy rumbles ending in a soft hum",
    "duration": 9.5
  }
]''')
MODEL='cvssp/audioldm-s-full-v2'
pipe=AudioLDMPipeline.from_pretrained(MODEL,torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
manifest=[]; log=[]
for a in ANIMALS:
    filename=f"call-{a['id']:02d}.wav"; duration=a['duration']; seed=20000+a['id']
    prompt='Close recording of a fictional jungle animal: '+a['soundPrompt']+'. Isolated creature vocalization, organic texture, clear foreground, no music or human speech.'
    started=time.time()
    output=pipe(prompt,negative_prompt='music, speech, humans, distortion, clipping, low quality',audio_length_in_s=duration,num_inference_steps=50,guidance_scale=2.5,generator=torch.Generator('cpu').manual_seed(seed)).audios[0]
    sr=16000; output=np.asarray(output,dtype=np.float32).reshape(-1)
    # Pad/crop to the precise requested duration and fade the boundaries.
    n=round(sr*duration); output=np.pad(output,(0,max(0,n-len(output))))[:n]
    peak=float(np.max(np.abs(output))); assert peak>1e-6, 'Silent output; rerun with a new seed.'
    output=output*(.85/peak)
    fade=min(int(.04*sr),n//2);output[:fade]*=np.linspace(0,1,fade);output[-fade:]*=np.linspace(1,0,fade)
    sf.write(ROOT/'sounds'/filename,output,sr,subtype='PCM_16')
    manifest.append(dict(specimen=a['id'],src='sounds/'+filename,duration=duration,seed=seed,model=MODEL))
    log.append(dict(specimen=a['id'],prompt=prompt,duration=duration,seed=seed,peak_before_normalization=peak,seconds=time.time()-started))
    (ROOT/'sound-manifest.json').write_text(json.dumps(manifest,indent=2))
    (ROOT/'sound-run.json').write_text(json.dumps(dict(model=MODEL,steps=50,guidance=2.5,sample_rate=sr,clips=log),indent=2))
    print(a['id'],a['name'],duration,'seconds',flush=True)
    display(Audio(filename=str(ROOT/'sounds'/filename)))
del pipe; gc.collect();torch.cuda.empty_cache()
assert len(manifest)==16 and len(set(x['duration'] for x in manifest))==16
print('SOUND GENERATION COMPLETE: sixteen WAV calls, 2 to 9.5 seconds.')

In [ ]:
import shutil
from google.colab import files
stage=Path('/content/sound-export');stage.mkdir(exist_ok=True)
shutil.copytree(ROOT/'sounds',stage/'sounds',dirs_exist_ok=True)
for name in ['sound-manifest.json','sound-run.json']:shutil.copy2(ROOT/name,stage/name)
shutil.make_archive('/content/jungle-sounds','zip',stage)
files.download('/content/jungle-sounds.zip')
